# IOAI — 2024 Summer National Hidden Treasure (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import numpy as np, os
from PIL import Image, ImageDraw, ImageFont
import matplotlib
fp=os.path.join(matplotlib.get_data_path(),'fonts/ttf/DejaVuSans-Bold.ttf')
font=ImageFont.truetype(fp,64); small=ImageFont.truetype(fp,28)
os.makedirs('data/hegyek',exist_ok=True)
comps=[('00_lat_deg','47'),('01_lat_min','28504'),('02_lon_deg','19'),('03_lon_min','3598')]
for i,(name,v) in enumerate(comps):
    rng=np.random.RandomState(100+i)
    bg=(rng.rand(240,420,3)*55+rng.randint(70,150)).astype(np.uint8)
    img=Image.fromarray(bg); d=ImageDraw.Draw(img); d.text((50,90),v,fill=(15,15,15),font=font)
    img.rotate(180).save(f'data/hegyek/{name}.jpg',quality=92)
for j,word in enumerate(['HEGYEK','TATRA']):
    rng=np.random.RandomState(200+j)
    bg=(rng.rand(240,420,3)*55+rng.randint(70,150)).astype(np.uint8)
    img=Image.fromarray(bg); d=ImageDraw.Draw(img); d.text((60,100),word,fill=(15,15,15),font=small)
    img.rotate(180).save(f'data/hegyek/zz_dist{j}.jpg',quality=92)
print('hegyek images:', sorted(os.listdir('data/hegyek')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Hidden Treasure (Elrejtett kincs)

> **HAIO 2024 — Summer National Finals (CV) · 15 pts**

János found an old book of nature photos with **strange characters printed on the pictures**. Read the characters and **decode the geographic coordinate** they spell out.

Use **EasyOCR** on the images under `data/hegyek/` (4 coordinate images + 2 scenery distractors), extract the numbers, and assemble the coordinate. Score = fraction of the 4 components correct (`lat_deg, lat_min, lon_deg, lon_min`). Expected answer: `N 47° 28,504'  E 19° 3,598'`.

⚠️ **The naive baseline below just runs OCR on each image as-is — and the reads come out empty or wrong (it scores ~0).** Look closely at the photos: *how are the characters oriented?* Figure out the transform the images need before OCR, then fix `run_inference`. (See the model solution for the intended approach.)

**Data note:** the organisers' original photos (`hegyek.zip`) were deleted, so the *same OCR-decode task* is reconstructed with bundled images. **Submit** `submission.csv` (`id,value`).

In [ ]:
!pip install -qq easyocr

In [ ]:
import os, re, glob
import numpy as np
import easyocr
from PIL import Image
reader = easyocr.Reader(['en'])  # downloads the OCR models on first run

In [ ]:
# --- NAIVE BASELINE: OCR each image exactly as stored (no preprocessing) ---
# TODO: the characters are printed in an unusual orientation -> add the right transform here.
def run_inference(img):
    # img: PIL.Image. Returns the highest-confidence numeric string ('' if none).
    # >>> your fix goes here (e.g. reorient the image before reading) <<<
    res = reader.readtext(np.array(img))
    cand = [(t, c) for (_, t, c) in res if any(ch.isdigit() for ch in t) and c > 0.5]
    return re.sub(r'\D', '', max(cand, key=lambda x: x[1])[0]) if cand else ''

paths = sorted(glob.glob('data/hegyek/*.jpg') + glob.glob('data/hegyek/*.jpeg') + glob.glob('data/hegyek/*.png'))
numbers = []
for p in paths:
    img = Image.open(p).convert('RGB')
    n = run_inference(img)
    if n:
        numbers.append(n)
    print(f'{os.path.basename(p)}: {n!r}')
print('decoded numbers:', numbers)

In [ ]:
import pandas as pd
# sorted coordinate images map to lat_deg, lat_min, lon_deg, lon_min
fields = ['lat_deg', 'lat_min', 'lon_deg', 'lon_min']
vals = (numbers + ['', '', '', ''])[:4]   # naive run usually leaves these wrong/empty -> low score
pd.DataFrame({'id': fields, 'value': vals}).to_csv('submission.csv', index=False)
print('wrote submission.csv:', dict(zip(fields, vals)))
if all(v.isdigit() for v in vals):
    print(f"Final Coordinate is:\n N 47\u00b0 {int(vals[1]):,}' \n E 19\u00b0 {int(vals[3]):,}'".replace(',', ' '))
else:
    print('\n(reads incomplete — fix run_inference so the characters are readable)')

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)